# Lab: Coding Regularization and Hyperparameter Tuning

## Lab goals

This lab prepares you for Problem Set 5 with a different application: hourly
bike-share demand. The lecture notes explain what regularization does. Here we focus
on making the Python workflow for preprocessing, pipelines, tuning, evaluation, and
coefficient recovery predictable.

By the end of the lab, you should be able to:

- screen numerical features with correlations while preserving coefficient signs;
- combine numerical and categorical preprocessing in a `Pipeline`;
- tune ridge and LASSO without leaking validation or test information;
- compare unregularized, ridge, and LASSO regression with RMSE and MAE;
- recover transformed feature names and coefficients from fitted pipelines;
- optionally extend the same workflow to L1 and L2 logistic regression using the
  inverse-strength parameter `C`; and
- diagnose common pipeline and model-selection mistakes.

Read the explanation before each code cell. You can run the notebook in Colab or
from the repository root. Restart the kernel and run all cells before relying on
the results.

## 0. Setup

In [ ]:
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")

The new tools have distinct roles:

- `ColumnTransformer` sends named numerical and categorical columns through
  different transformations.
- `Pipeline` treats preprocessing plus model fitting as one estimator.
- `GridSearchCV` refits that complete estimator for every parameter value and fold.
- `Ridge` and `Lasso` fit L2- and L1-regularized linear models.
- `LogisticRegression` supports the same penalty families for a binary outcome.
- `mean_absolute_error()` averages absolute prediction errors; unlike RMSE, it does
  not square errors before averaging.

## 1. Helper functions

`course_data_source()` first searches for the course CSV locally and otherwise
returns its public GitHub URL, so the same code works in Colab.
`make_one_hot_encoder()` supports the keyword used by both current and older
scikit-learn releases. `rmse()` returns error in the target's units.
`clean_feature_name()` removes the prefixes that `ColumnTransformer` adds.

In [ ]:
PUBLIC_REPOSITORY = "okuchap/GB656_2026_public"
PUBLIC_REVISION = "main"


def course_data_source(file_name):
    """Return a local course-data path when available, otherwise its public URL."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local_path = root / "data" / file_name
        if local_path.is_file():
            return local_path

    encoded_name = quote(file_name)
    return (
        "https://raw.githubusercontent.com/"
        f"{PUBLIC_REPOSITORY}/{PUBLIC_REVISION}/data/{encoded_name}"
    )


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def clean_feature_name(feature_name):
    return (
        feature_name
        .replace("num__", "")
        .replace("cat__", "")
        .replace("remainder__", "")
    )

## 2. Load and inspect the bike-demand data

Each row of `SeoulBikeData.csv` is one hour. We rename columns once so later
feature lists are short and exact. Parsing `date` unlocks calendar features;
converting `hour` to text tells the encoder to treat hours as categories rather than
assume a straight-line numeric effect.

In [ ]:
bike_source = course_data_source("SeoulBikeData.csv")
bike_raw = pd.read_csv(bike_source, encoding="latin1")

bike = bike_raw.rename(
    columns={
        "Date": "date",
        "Rented Bike Count": "bike_count",
        "Hour": "hour",
        "Temperature(°C)": "temperature",
        "Humidity(%)": "humidity",
        "Wind speed (m/s)": "wind_speed",
        "Visibility (10m)": "visibility",
        "Dew point temperature(°C)": "dew_point",
        "Solar Radiation (MJ/m2)": "solar_radiation",
        "Rainfall(mm)": "rainfall",
        "Snowfall (cm)": "snowfall",
        "Seasons": "season",
        "Holiday": "holiday",
        "Functioning Day": "functioning_day",
    }
)

bike["date"] = pd.to_datetime(bike["date"], dayfirst=True)
bike["month"] = bike["date"].dt.month.astype(str)
bike["day_of_week"] = bike["date"].dt.day_name()
bike["hour"] = bike["hour"].astype(str)

bike = bike[bike["functioning_day"] == "Yes"].copy()
bike = bike.sort_values(["date", "hour"]).reset_index(drop=True)

bike.head()

We filter to functioning days because a closure's zero rentals do not represent
ordinary customer demand. The summary checks the target scale and several weather
inputs. The histogram is a brief reminder that `ax.hist()` groups a numerical
variable into intervals; `bins=30` controls the number of displayed intervals.

In [ ]:
bike[["bike_count", "temperature", "humidity", "rainfall", "snowfall"]].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(bike["bike_count"], bins=30, edgecolor="white")
ax.set_xlabel("Rented bikes per hour")
ax.set_ylabel("Number of hours")
ax.set_title("Distribution of hourly bike demand")

plt.show()

## 3. Create features and make a correlation screen

`add_bike_features()` starts with `.copy()` so it does not modify its input. It adds
a categorical hour-by-season feature and two numerical interactions. These related
columns give regularization useful work to do.

The numerical and categorical lists are the single source of truth for
`ColumnTransformer`. A name missing from both lists will not reach the model; a
misspelling raises an error during fitting.

In [ ]:
def add_bike_features(data):
    data = data.copy()

    data["hour_season"] = data["hour"].astype(str) + "_" + data["season"].astype(str)
    data["temperature_summer"] = (
        data["temperature"] * (data["season"] == "Summer").astype(int)
    )
    data["rainfall_commute"] = (
        data["rainfall"] * data["hour"].astype(int).isin([7, 8, 17, 18]).astype(int)
    )

    return data


bike_model = add_bike_features(bike)

numeric_features = [
    "temperature",
    "humidity",
    "wind_speed",
    "visibility",
    "dew_point",
    "solar_radiation",
    "rainfall",
    "snowfall",
    "temperature_summer",
    "rainfall_commute",
]

categorical_features = [
    "hour",
    "month",
    "day_of_week",
    "season",
    "holiday",
    "hour_season",
]

feature_columns = numeric_features + categorical_features

X = bike_model[feature_columns]
y = bike_model["bike_count"]

X.head()

`DataFrame.corrwith(y)` calculates one Pearson correlation between every numerical
column and `y`. We preserve the signed correlation for interpretation, add an
absolute-value column only for ranking, and then display the five strongest marginal
relationships. This is an exploratory screen, not a causal analysis or a substitute
for cross-validated model comparison.

In [ ]:
correlation_screen = (
    bike_model[numeric_features]
    .corrwith(y)
    .rename("correlation")
    .rename_axis("feature")
    .reset_index()
    .assign(abs_correlation=lambda data: data["correlation"].abs())
    .sort_values("abs_correlation", ascending=False)
)

correlation_screen.head(5).round(3)

## 4. Make a train-test split

The test set is held out until all tuning is complete. `random_state=606` makes the
split reproducible. We may display split sizes now, but we do not use test outcomes
to choose preprocessing, a grid, or a model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=606,
)

pd.Series(
    {"Training rows": len(X_train), "Held-out test rows": len(X_test)},
    name="rows",
)

## 5. Build preprocessing and model pipelines

Numerical variables have different units, so `StandardScaler` centers and scales
them. Categorical variables require one-hot columns; `handle_unknown="ignore"`
prevents a validation or test category unseen during fitting from causing an error.

`ColumnTransformer` combines those operations into one fitted recipe. Crucially,
when it is inside a pipeline used by cross-validation, every fold learns scaling
statistics and category levels from that fold's training rows only.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", make_one_hot_encoder(), categorical_features),
    ]
)

Each pipeline uses stable step names: `preprocess` and `model`. A grid key such as
`model__alpha` means “set `alpha` on the object in the `model` step.” A wrong step or
parameter name produces a parameter-not-found error.

Standardization is especially important for ridge and LASSO because their penalties depend on coefficient magnitudes. If predictors are measured on very different scales, their coefficients are also naturally on different scales, so the penalty would not treat predictors comparably. Standardizing the numerical predictors before fitting puts them on a common scale. Keeping `StandardScaler` **inside** the pipeline is equally important. During cross-validation, the scaler is then fitted separately using only the training portion of each fold, preventing information from the validation fold from leaking into preprocessing.

We leave the one-hot columns as zero/one indicators. LASSO receives a generous iteration limit to support convergence.

In [ ]:
ols_pipeline = Pipeline(
    steps=[("preprocess", preprocessor), ("model", LinearRegression())]
)

ridge_pipeline = Pipeline(
    steps=[("preprocess", preprocessor), ("model", Ridge())]
)

lasso_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", Lasso(max_iter=100000, tol=0.001)),
    ]
)

## 6. Tune ridge and LASSO

`KFold` creates five reproducible train/validation partitions.
`np.logspace(-2, 3, 11)` supplies penalty strengths from 0.01 to 1,000 in
multiplicative steps.

Scikit-learn maximizes every scorer, so it stores losses under names beginning with
`neg_`. The positive mean CV-RMSE is therefore `-search.best_score_`.
`best_params_` reports the chosen value and `best_estimator_` is already refit on the
full training set.

**What does `alpha` control?**

For both ridge and LASSO, `alpha` controls the strength of regularization. A small `alpha` places relatively little penalty on large coefficients, so the fitted model behaves more like ordinary linear regression. As `alpha` increases, the penalty becomes stronger and coefficients are pushed more strongly toward zero. Ridge typically shrinks coefficients without setting them exactly to zero, while LASSO can shrink some coefficients all the way to zero.

For each candidate `alpha`, `GridSearchCV`:

1. fits the entire pipeline on four training folds;
2. evaluates it on the remaining validation fold;
3. repeats this across all five folds;
4. averages the validation scores;
5. compares the candidate `alpha` values; and
6. refits the pipeline on the full training set using the selected alpha.

Note: The held-out test set is not involved in any of these steps.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=606)
alpha_grid = np.logspace(-2, 3, 11)

ridge_search = GridSearchCV(
    ridge_pipeline,
    param_grid={"model__alpha": alpha_grid},
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=1,
)

lasso_search = GridSearchCV(
    lasso_pipeline,
    param_grid={"model__alpha": alpha_grid},
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=1,
)

ridge_search.fit(X_train, y_train)
lasso_search.fit(X_train, y_train);

A selected value at either grid endpoint is a diagnostic, not proof of an invalid
model. It means the best value in the attempted grid sits against a boundary, so the
grid should be extended using training data only before the test set is examined.

In [ ]:
tuning_summary = pd.DataFrame(
    {
        "model": ["Ridge", "LASSO"],
        "selected_alpha": [
            ridge_search.best_params_["model__alpha"],
            lasso_search.best_params_["model__alpha"],
        ],
        "mean_cv_rmse": [-ridge_search.best_score_, -lasso_search.best_score_],
    }
)

tuning_summary["at_grid_endpoint"] = tuning_summary["selected_alpha"].isin(
    [alpha_grid.min(), alpha_grid.max()]
)

tuning_summary.round(3)

## 7. Compare regression models

Tuning is complete, so we now fit the unregularized pipeline and use the test set
once. The comparison repeats two earlier ideas—training RMSE and cross-validated
RMSE—and adds test MAE. RMSE gives extra weight to large errors; MAE is the average
absolute error. Both remain in rented bikes per hour.

The selected ridge and LASSO CV scores come directly from the searches. For OLS,
`cross_val_score()` refits the complete pipeline in the same folds.

In [ ]:
ols_pipeline.fit(X_train, y_train)

ols_mean_cv_rmse = -cross_val_score(
    ols_pipeline,
    X_train,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=1,
).mean()

regression_models = [
    ("OLS", ols_pipeline, ols_mean_cv_rmse),
    ("Ridge tuned by CV", ridge_search.best_estimator_, -ridge_search.best_score_),
    ("LASSO tuned by CV", lasso_search.best_estimator_, -lasso_search.best_score_),
]

comparison_rows = []

for model_name, fitted_model, mean_cv_rmse in regression_models:
    training_prediction = fitted_model.predict(X_train)
    test_prediction = fitted_model.predict(X_test)
    comparison_rows.append(
        {
            "model": model_name,
            "training_rmse": rmse(y_train, training_prediction),
            "mean_cv_rmse": mean_cv_rmse,
            "test_rmse": rmse(y_test, test_prediction),
            "test_mae": mean_absolute_error(y_test, test_prediction),
        }
    )

regression_comparison = pd.DataFrame(comparison_rows)
regression_comparison.round(2)

Read the table rather than assuming that regularization must win. A test-set result
can differ from its CV ranking because each estimate uses a finite set of held-out
observations. Do not change the grid after seeing the test results; that would turn
the test set into another validation set.

## 8. Recover transformed feature names and coefficients

After preprocessing, the columns seen by the regression model may differ from the original columns. For example, one categorical variable can become several one-hot columns. Therefore, coefficients must be paired with the feature names produced by the **same fitted preprocessor** that produced the model's input matrix. Otherwise, names and coefficients may be misaligned.

The fitted preprocessor expands categorical inputs into indicator columns.
`get_feature_names_out()` retrieves names in exactly the order supplied to the fitted
model. Names and coefficients must come from the same selected pipeline, or their
lengths and positions may not match.

In [ ]:
coefficient_tables = []

for model_name, fitted_pipeline in [
    ("Ridge", ridge_search.best_estimator_),
    ("LASSO", lasso_search.best_estimator_),
]:
    transformed_names = [
        clean_feature_name(name)
        for name in fitted_pipeline.named_steps["preprocess"].get_feature_names_out()
    ]
    coefficients = fitted_pipeline.named_steps["model"].coef_

    coefficient_tables.append(
        pd.DataFrame(
            {
                "model": model_name,
                "feature": transformed_names,
                "coefficient": coefficients,
            }
        )
    )

coefficient_table = pd.concat(coefficient_tables, ignore_index=True)
coefficient_table.head()

Count active coefficients using a small tolerance because floating-point algorithms
can produce tiny numerical values. The LASSO count describes this fitted sample and
chosen alpha; correlated features can exchange roles in another sample.

In [ ]:
coefficient_counts = (
    coefficient_table
    .assign(nonzero=lambda data: data["coefficient"].abs() > 1e-8)
    .groupby("model", as_index=False)
    .agg(
        total_coefficients=("coefficient", "size"),
        nonzero_coefficients=("nonzero", "sum"),
    )
)

coefficient_counts["zero_coefficients"] = (
    coefficient_counts["total_coefficients"]
    - coefficient_counts["nonzero_coefficients"]
)

coefficient_counts

Sort on magnitude to find large positive and negative coefficients while keeping the
original sign. Grouping after sorting returns the ten largest magnitudes within each
model.

In [ ]:
top_coefficients = (
    coefficient_table
    .assign(abs_coefficient=lambda data: data["coefficient"].abs())
    .sort_values(["model", "abs_coefficient"], ascending=[True, False])
    .groupby("model", as_index=False)
    .head(10)
)

top_coefficients.round(2)

Because numerical inputs were standardized, a numerical coefficient is the predicted
rental difference for a one-standard-deviation increase, holding the other
transformed columns fixed. One-hot coefficients compare indicator states within the
encoded design. These are predictive conditional associations, not causal effects.

## 9. Optional extension: tune regularized logistic regression

**This section is enrichment and is not required for Problem Set 5**. Complete Sections 1–8 first. The main idea transfers from linear regression, but scikit-learn parameterizes logistic regularization using `C`, where **smaller `C` means stronger regularization**. To practice the same workflow for
classification, we create a binary outcome:
whether demand is at or above the sample median. `stratify=y_binary` approximately
preserves the class share in the two subsets, a method introduced in the
classification lab.

In [ ]:
y_binary = (bike_model["bike_count"] >= bike_model["bike_count"].median()).astype(int)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X,
    y_binary,
    test_size=0.25,
    random_state=606,
    stratify=y_binary,
)

y_binary.value_counts(normalize=True).sort_index()

For `LogisticRegression`, smaller `C` means stronger regularization—the inverse
direction from `alpha`. With the current scikit-learn API, `l1_ratio=0.0` requests
pure L2 and `l1_ratio=1.0` requests pure L1. We use `lbfgs` for L2 and
`saga` for L1. Both leave the intercept unpenalized, matching the lecture equations.
The larger iteration limit supports convergence; `random_state` makes SAGA
reproducible.

We tune negative log loss because it evaluates probability quality. At the end,
`predict_proba(X)[:, 1]` selects the probability of class 1. Log loss is
lower-is-better; ROC AUC is higher-is-better and evaluates ranking rather than the
numerical accuracy of probabilities.

In [ ]:
C_grid = np.logspace(-2, 2, 7)

l2_logit_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            LogisticRegression(
                l1_ratio=0.0,
                solver="lbfgs",
                max_iter=10000,
                random_state=606,
            ),
        ),
    ]
)

l1_logit_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            LogisticRegression(
                l1_ratio=1.0,
                solver="saga",
                max_iter=10000,
                random_state=606,
            ),
        ),
    ]
)

l2_logit_search = GridSearchCV(
    l2_logit_pipeline,
    param_grid={"model__C": C_grid},
    scoring="neg_log_loss",
    cv=cv,
    n_jobs=1,
)

l1_logit_search = GridSearchCV(
    l1_logit_pipeline,
    param_grid={"model__C": C_grid},
    scoring="neg_log_loss",
    cv=cv,
    n_jobs=1,
)

l2_logit_search.fit(X_train_clf, y_train_clf)
l1_logit_search.fit(X_train_clf, y_train_clf);

In [ ]:
logit_rows = []

for model_name, search in [
    ("L2 logistic regression", l2_logit_search),
    ("L1 logistic regression", l1_logit_search),
]:
    positive_probability = search.best_estimator_.predict_proba(X_test_clf)[:, 1]
    logit_rows.append(
        {
            "model": model_name,
            "selected_C": search.best_params_["model__C"],
            "test_log_loss": log_loss(y_test_clf, positive_probability),
            "test_auc": roc_auc_score(y_test_clf, positive_probability),
        }
    )

pd.DataFrame(logit_rows).round(3)

## 10. Common mistakes and quick diagnoses

| Symptom | Likely cause | Check or fix |
|---|---|---|
| Grid reports an invalid parameter | Step/key mismatch | Compare `model__alpha` or `model__C` with the pipeline's named steps |
| CV looks implausibly optimistic | Preprocessing was fit before CV | Put the unfitted transformer inside the pipeline |
| Negative displayed RMSE or log loss | Scoring convention was not reversed | Report `-best_score_` for a loss |
| Feature-name and coefficient lengths differ | Names came from another or unfitted preprocessor | Retrieve both from the same `best_estimator_` |
| A selected value equals a grid endpoint | Search range may be too narrow | Extend the grid and tune again using training data only |
| Convergence warning | Optimization has not stabilized | Check scaling, tolerance, iteration limit, and extreme grid values |
| Test set influences a later choice | Test leakage | Freeze choices before the first test prediction |

## 11. Transfer to Problem Set 5

For the residential-building task, the raw predictors are already numerical, so a
`StandardScaler` and model can form a simpler two-step pipeline. The rest of the
workflow transfers directly:

1. define the target, prediction time, and permissible features;
2. split once and set the test data aside;
3. put scaling and the estimator in one pipeline;
4. tune on training data with reproducible folds and a fixed grid;
5. freeze the choice and compare once on the test data;
6. recover feature names and coefficients from the same fitted pipeline; and
7. restart the kernel and run every cell to expose hidden state.

If you can explain why each step protects the evaluation boundary or prevents a
shape/name mismatch, you are ready to adapt the workflow.